# Jury LLM: Multi-Model & Human-in-the-Loop Evaluation System

This notebook serves as the interactive frontend for the Jury System.

In [2]:
import sys
import os
sys.path.append('../')

from dotenv import load_dotenv
load_dotenv('../.env')

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
from src.utils import parse_json_output
from src.llm_provider import LLMProvider
from src.agents import QUALIFICATION_PROMPT
from src.qualification import QualificationFlow
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

llm = LLMProvider()
flow = QualificationFlow()

INFO:src.llm_provider:Using DashScope configuration.
INFO:src.llm_provider:Using DashScope configuration.


## Step 1: Input Target Text
Please paste the LLM generated text you want to evaluate below. This will be stored globally for the entire session.

In [4]:
import os
# Fix for 502 Bad Gateway / Proxy issues with Gradio
os.environ['no_proxy'] = 'localhost,127.0.0.1'
if 'http_proxy' in os.environ: del os.environ['http_proxy']
if 'https_proxy' in os.environ: del os.environ['https_proxy']

import gradio as gr

# --- Global Variables ---
TARGET_TEXT = ""
EVALUATION_PURPOSE = "General Assessment"
HUMAN_COMPETENCY_SCORE = 0
QUALIFICATION_HISTORY = []
MAX_ROUNDS = 3  # Checkpoint had 3 rounds
CURRENT_ROUND = 0

def save_step1_inputs(text, purpose):
    global TARGET_TEXT, EVALUATION_PURPOSE
    
    if not text.strip():
        return "❌ Error: Target Text cannot be empty."
    
    TARGET_TEXT = text
    EVALUATION_PURPOSE = purpose
    flow.set_target_text(text, purpose)
    
    return f'''✅ Saved Successfully!
    
    Target Text Length: {len(text)} chars
    Evaluation Purpose: {purpose}
    
    You can now stop this cell and run Step 2.'''

with gr.Blocks() as step1_demo:
    gr.Markdown("## Step 1: Input Target Text")
    gr.Markdown("Please paste the LLM generated text you want to evaluate below and specify your evaluation goal.")
    
    txt_input = gr.Textbox(
        label="Target Text", 
        lines=10, 
        placeholder="Paste the LLM generated text here..."
    )
    
    purpose_input = gr.Textbox(
        label="Evaluation Purpose",
        value="General Assessment",
        lines=2,
        placeholder="e.g. Verify legal accuracy, Check for creative writing style, etc."
    )
    
    submit_btn = gr.Button("Confirm & Save Text", variant="primary")
    output_msg = gr.Markdown()
    
    submit_btn.click(
        fn=save_step1_inputs, 
        inputs=[txt_input, purpose_input], 
        outputs=output_msg
    )

print("Launching Step 1 Interface...")
try:
    step1_demo.launch(height=600, inline=True, quiet=False)
except Exception as e:
    print(f"Error launching Gradio: {e}")


INFO:httpx:HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


Launching Step 1 Interface...
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Step 2: Qualification Assessment
The AI will now analyze the text you provided in Step 1 and assess your qualification to evaluate it.

In [8]:
import os
# Fix for 502 Bad Gateway / Proxy issues with Gradio
os.environ['no_proxy'] = 'localhost,127.0.0.1'
if 'http_proxy' in os.environ: del os.environ['http_proxy']
if 'https_proxy' in os.environ: del os.environ['https_proxy']

import gradio as gr
print(f'📦 Gradio version: {gr.__version__}')

with gr.Blocks() as step2_demo:
    gr.Markdown("## Step 2: Qualification Assessment")
    gr.Markdown("""### Choose Your Assessment Method:
    
**Option 1: AI Interview** - Let the AI assess your qualification through interactive questions.  
**Option 2: Self Rating** - Directly provide your competency score (0-100) to skip the interview.
    """)
    
    with gr.Tabs():
        with gr.Tab("🤖 AI Interview"):
            gr.Markdown("""The AI will ask you questions to evaluate your competency.  
Set the maximum number of rounds and click **Start AI Assessment**.""")
            
            max_rounds_input = gr.Slider(
                minimum=3, 
                maximum=50, 
                value=15, 
                step=1, 
                label="Max Rounds",
                info="Maximum number of questions the AI can ask"
            )
            
            with gr.Row():
                start_ai_btn = gr.Button("🚀 Start AI Assessment", variant="primary", scale=2)
                clear_btn = gr.ClearButton(value="🔄 Reset", scale=1)
            
            chatbot = gr.Chatbot(height=500, label="Interview Chat")
            msg = gr.Textbox(
                label="Your Answer", 
                placeholder="Type your answer here and press Enter...", 
                interactive=False
            )
            
            # Hidden input for AI mode (self_rating = 0)
            ai_self_rating = gr.Number(value=0, visible=False)
            
            # AI Interview Event Handlers
            start_ai_btn.click(
                flow.on_start_interview, 
                inputs=[max_rounds_input, ai_self_rating],
                outputs=[chatbot, msg]
            )
            msg.submit(flow.on_user_reply, [msg, chatbot], [chatbot, msg])
            clear_btn.click(lambda: None, None, chatbot, queue=False)
        
        with gr.Tab("✍️ Self Rating"):
            gr.Markdown("""If you already know your competency level, you can directly provide a score.  
This will skip the AI interview process entirely.""")
            
            gr.Markdown("### Score Guidelines:")
            gr.Markdown("""
- **0-30**: Layperson (No professional background)
- **31-60**: Hobbyist (Some knowledge or interest)
- **61-85**: Professional (Relevant expertise)
- **86-100**: Expert (Deep domain knowledge)
            """)
            
            self_rating_input = gr.Slider(
                minimum=0,
                maximum=100,
                value=50,
                step=1,
                label="Your Competency Score",
                info="Rate your qualification level (0-100)"
            )
            
            self_rating_reason = gr.Textbox(
                label="Reason (Optional)",
                placeholder="Why do you rate yourself at this level?",
                lines=3
            )
            
            submit_self_rating_btn = gr.Button("✅ Submit Self Rating", variant="primary")
            self_rating_output = gr.Chatbot(height=200, label="Assessment Result")
            
            # Hidden inputs for self-rating mode
            dummy_max_rounds = gr.Number(value=3, visible=False)
            dummy_msg = gr.Textbox(visible=False)
            
            # Self Rating Event Handler
            submit_self_rating_btn.click(
                flow.on_start_interview,
                inputs=[dummy_max_rounds, self_rating_input],
                outputs=[self_rating_output, dummy_msg]
            )

print('🚀 Launching Step 2 Gradio Interface...')
try:
    # Launch with explicit parameters for Jupyter
    step2_demo.launch(
        height=700,
        inline=True,
        share=False,
        debug=False,
        show_error=True
    )
    print('✅ Step 2 interface launched successfully!')
except Exception as e:
    print(f'❌ Error launching Gradio: {e}')
    import traceback
    traceback.print_exc()


📦 Gradio version: 6.1.0
🚀 Launching Step 2 Gradio Interface...
* Running on local URL:  http://127.0.0.1:7861


INFO:httpx:HTTP Request: GET http://127.0.0.1:7861/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7861/ "HTTP/1.1 200 OK"


* To create a public link, set `share=True` in `launch()`.


✅ Step 2 interface launched successfully!


## Step 2.5: Evaluation Criteria (Rubrics)
Define the evaluation criteria that will be used by all judges in Step 3.

You can either:
1. **Provide your own criteria** (paste into the text box)
2. **Generate criteria using AI** (leave text box empty and click Generate)

In [11]:
import os
# Fix for 502 Bad Gateway / Proxy issues with Gradio
os.environ['no_proxy'] = 'localhost,127.0.0.1'
if 'http_proxy' in os.environ: del os.environ['http_proxy']
if 'https_proxy' in os.environ: del os.environ['https_proxy']

import gradio as gr
from src.rubrics import RubricsFlow

# --- Global Variable for Rubrics ---
EVALUATION_RUBRICS = ""

# Initialize rubrics flow (lazy initialization)
rubrics_flow = None

def get_rubrics_flow():
    """Lazy initialization of RubricsFlow to ensure TARGET_TEXT is available."""
    global rubrics_flow
    if rubrics_flow is None:
        rubrics_flow = RubricsFlow()
    return rubrics_flow

def on_use_manual_rubrics(manual_input):
    """User provides their own rubrics."""
    global EVALUATION_RUBRICS
    
    if not manual_input or not manual_input.strip():
        return "❌ Error: Please paste your evaluation criteria.", "", gr.update(interactive=False)
    
    EVALUATION_RUBRICS = manual_input.strip()
    rf = get_rubrics_flow()
    rf.store_rubrics(EVALUATION_RUBRICS)
    
    return (
        f"✅ Your criteria have been saved! ({len(EVALUATION_RUBRICS)} characters)",
        gr.update(value=EVALUATION_RUBRICS, interactive=True),
        gr.update(interactive=True)
    )

def on_generate_rubrics(manual_input):
    """Generate rubrics using AI."""
    global EVALUATION_RUBRICS
    
    # Only generate if manual input is empty
    if manual_input and manual_input.strip():
        return (
            "⚠️ Manual criteria detected. Please clear the manual input first or click 'Use Manual Criteria'.",
            manual_input,
            gr.update(interactive=False)
        )
    
    if not TARGET_TEXT:
        return "❌ Error: No target text found. Please complete Step 1 first.", "", gr.update(interactive=False)
    
    # Initialize rubrics flow with current context
    rf = get_rubrics_flow()
    rf.set_context(TARGET_TEXT, EVALUATION_PURPOSE)
    
    # Generate rubrics
    generated = rf.generate_rubrics()
    
    if generated.startswith("❌"):
        return generated, "", gr.update(interactive=False)
    
    return (
        "✅ Criteria generated! Please review and edit if needed, then click 'Confirm and Continue'.",
        gr.update(value=generated, interactive=True),
        gr.update(interactive=True)
    )

def on_confirm_rubrics(edited_rubrics):
    """Confirm and store the final rubrics."""
    global EVALUATION_RUBRICS
    
    if not edited_rubrics or not edited_rubrics.strip():
        return "❌ Error: Cannot save empty criteria."
    
    EVALUATION_RUBRICS = edited_rubrics.strip()
    rf = get_rubrics_flow()
    rf.store_rubrics(EVALUATION_RUBRICS)
    
    preview = EVALUATION_RUBRICS[:500] + "..." if len(EVALUATION_RUBRICS) > 500 else EVALUATION_RUBRICS
    
    return f"""✅ **Evaluation Criteria Confirmed!**

📊 **Criteria Length:** {len(EVALUATION_RUBRICS)} characters

**Preview:**
```
{preview}
```

✨ You can now stop this cell and proceed to **Step 3**."""

with gr.Blocks() as step25_demo:
    gr.Markdown("## Step 2.5: Evaluation Criteria (Rubrics)")
    gr.Markdown("""### Define Evaluation Standards
    
Establish the criteria that all judges (AI models and human) will use in Step 3.  
Choose between **manual input** or **AI generation**.
    """)
    
    with gr.Tabs():
        with gr.Tab("✍️ Manual Input"):
            gr.Markdown("""Provide your own evaluation criteria in markdown format.  
This gives you full control over the rubrics.""")
            
            manual_input = gr.Textbox(
                label="Your Evaluation Criteria",
                lines=12,
                placeholder="""Example:
# Evaluation Criteria

## 1. Accuracy (40 points)
- Factual correctness
- No misleading information

## 2. Clarity (30 points)
- Easy to understand
- Well-structured

## 3. Completeness (30 points)
- Addresses all aspects
- Sufficient detail
""",
                info="Paste or type your criteria here"
            )
            
            use_manual_btn = gr.Button("✅ Use Manual Criteria", variant="primary")
        
        with gr.Tab("🤖 AI Generation"):
            gr.Markdown("""Let AI generate evaluation criteria based on your target text and evaluation purpose.  
The AI will analyze the content and create appropriate rubrics.""")
            
            gr.Markdown("### Current Context:")
            # Update context display dynamically
            context_info = f"""- **Target Text Length:** {len(TARGET_TEXT) if TARGET_TEXT else 0} characters  
- **Evaluation Purpose:** {EVALUATION_PURPOSE if EVALUATION_PURPOSE else 'Not set'}"""
            context_display = gr.Markdown(context_info)
            
            gr.Markdown("### Instructions:")
            gr.Markdown("""1. Ensure the **Manual Input** tab is empty
2. Click **Generate Criteria** below
3. Review the generated criteria in the editor
4. Make any necessary edits
5. Click **Confirm and Continue**""")
            
            generate_btn = gr.Button("🚀 Generate Evaluation Criteria", variant="primary")
    
    status_msg = gr.Markdown()
    
    gr.Markdown("---")
    gr.Markdown("### 📝 Review & Edit Criteria")
    gr.Markdown("Your criteria will appear below. You can edit them before confirming.")
    
    edit_area = gr.Textbox(
        label="Final Criteria (Editable)",
        lines=15,
        placeholder="Your criteria will appear here for review and editing...",
        interactive=False
    )
    
    confirm_btn = gr.Button(
        "✅ Confirm and Continue to Step 3", 
        variant="primary",
        interactive=False
    )
    final_msg = gr.Markdown()
    
    # Event handlers
    use_manual_btn.click(
        fn=on_use_manual_rubrics,
        inputs=[manual_input],
        outputs=[status_msg, edit_area, confirm_btn]
    )
    
    generate_btn.click(
        fn=on_generate_rubrics,
        inputs=[manual_input],
        outputs=[status_msg, edit_area, confirm_btn]
    )
    
    confirm_btn.click(
        fn=on_confirm_rubrics,
        inputs=[edit_area],
        outputs=[final_msg]
    )

print("Launching Step 2.5 Interface...")
try:
    step25_demo.launch(height=800, inline=True, quiet=False, debug=False, show_error=True)
except Exception as e:
    print(f"Error launching Gradio: {e}")
    import traceback
    traceback.print_exc()


Launching Step 2.5 Interface...


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://127.0.0.1:7862/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7862/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## Step 3: Jury Evaluation
Now that your qualification is established, we proceed to the jury evaluation.

In [ ]:
import yaml
import gradio as gr
from IPython.display import display

# Initialize AgentScope with Studio (optional)
import agentscope
import os

# Connect to Studio if URL is provided
studio_url = os.getenv('AGENTSCOPE_STUDIO_URL', 'http://localhost:3000')
try:
    agentscope.init(
        project='Jury-LLM',
        studio_url=studio_url
    )
    print(f'✅ AgentScope Studio connected: {studio_url}')
except Exception as e:
    print(f'ℹ️  Studio not connected (this is optional): {e}')

# Load configuration
with open('../config/jury_config.yaml', 'r') as f:
    JURY_CONFIG = yaml.safe_load(f)

# Import AgentScope-based jury system
from src.agentscope_jury import JuryEvaluationSystem

# Initialize the jury system
print('🔧 Initializing Jury Evaluation System...')
jury_system = JuryEvaluationSystem(JURY_CONFIG)
print(f'✅ System ready with {len(jury_system.jury_agents)} jury members')

# Global variable to store intermediate state
JURY_STATE = None

def start_jury_evaluation(human_score, human_reason):
    """Run jury evaluation with human input"""
    global JURY_STATE, HUMAN_COMPETENCY_SCORE
    
    # Get competency score from previous step
    HUMAN_COMPETENCY_SCORE = flow.get_score()
    
    if not TARGET_TEXT:
        return "❌ Error: No Target Text found. Please complete Step 1."
    
    output_text = f"📊 Human Competency Score from Step 2: {HUMAN_COMPETENCY_SCORE:.2f}\n"
    output_text += f"📝 Your Evaluation Score: {human_score}\n"
    output_text += f"💬 Your Reason: {human_reason[:100]}...\n\n"
    
    if not EVALUATION_RUBRICS:
        output_text += "⚠️  Warning: No evaluation rubrics found.\n"
        output_text += "   Proceeding without rubrics...\n\n"
    
    try:
        # Run jury evaluation (up to voting point)
        result = jury_system.run(
            topic=TARGET_TEXT,
            human_score=human_score,
            human_reason=human_reason,
            human_bio=f"Competency Score: {HUMAN_COMPETENCY_SCORE:.2f} (Assessed via Qualification Exam)",
            rubrics=EVALUATION_RUBRICS
        )
        
        # Store state for continuation
        JURY_STATE = result['state']
        
        output_text += "\n" + "="*60 + "\n"
        output_text += "⏸️  SYSTEM PAUSED FOR HUMAN VOTE\n"
        output_text += "="*60 + "\n"
        output_text += "\n✅ Evaluation complete! Please proceed to Step 4 to cast your vote.\n"
        
        return output_text
        
    except Exception as e:
        import traceback
        error_msg = f"❌ Error during jury evaluation: {str(e)}\n\n"
        error_msg += traceback.format_exc()
        return error_msg

# Create Gradio interface
with gr.Blocks(title="Step 3: Jury Evaluation") as step3_interface:
    gr.Markdown("# Step 3: Jury Evaluation")
    gr.Markdown("Provide your evaluation score and reasoning for the target text.")
    
    with gr.Row():
        with gr.Column():
            score_slider = gr.Slider(
                minimum=0,
                maximum=100,
                value=80,
                step=1,
                label="Your Score"
            )
            reason_textbox = gr.Textbox(
                label="Your Reason",
                placeholder="Why did you give this score?",
                lines=5
            )
            submit_btn = gr.Button("Start Jury Evaluation", variant="primary")
        
        with gr.Column():
            output_box = gr.Textbox(
                label="Evaluation Output",
                lines=20,
                max_lines=30
            )
    
    submit_btn.click(
        fn=start_jury_evaluation,
        inputs=[score_slider, reason_textbox],
        outputs=output_box
    )

# Display the interface
step3_interface.launch(inline=True, share=False, debug=True)


2026-02-03 16:02:35,366 | INFO    | _user_input:on_connect:198 - Connected to AgentScope Studio at "http://localhost:3000" with run name "JDLSjgGgryMPKbHSyhBoHv".
2026-02-03 16:02:35,366 | INFO    | _user_input:on_connect:204 - View the run at: http://localhost:3000/projects/UnnamedProject_At20260203
INFO:src.llm_provider:Using DashScope configuration.


✅ AgentScope Studio connected: http://localhost:3000
🔧 Initializing Jury Evaluation System...
✅ System ready with 5 jury members


INFO:httpx:HTTP Request: GET http://127.0.0.1:7863/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7863/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


INFO:httpx:HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
/Users/Thomas/Desktop/Jury/jury-llm/juryenv/lib/python3.12/site-packages/gradio/routes.py:1350: DeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


🏛️  JURY EVALUATION SYSTEM (AgentScope)

📊 Step 1: Calculating Human Authority Weight...


## Step 4: Voting Interface

In [ ]:
import gradio as gr
from IPython.display import display

def show_voting_options():
    """Display anonymized voting options"""
    global JURY_STATE
    
    if JURY_STATE is None:
        return "❌ Error: No jury state found. Please complete Step 3 first.", {}
    
    # Get anonymized options
    anonymized_options = JURY_STATE.anonymized_reasons
    
    # Format for display
    display_text = "🎭 Anonymous Evaluation Options:\n"
    display_text += "="*60 + "\n\n"
    
    for opt_id, text in anonymized_options.items():
        display_text += f"{opt_id}:\n{text}\n"
        display_text += "-"*60 + "\n\n"
    
    # Create choices for radio
    choices = list(anonymized_options.keys())
    
    return display_text, gr.Radio(choices=choices, label="Select the best evaluation")

def submit_vote(selected_option):
    """Handle vote submission"""
    global JURY_STATE
    
    if not selected_option:
        return "❌ Error: Please select an option before submitting."
    
    output_text = f"✅ You voted for: {selected_option}\n\n"
    output_text += "🔄 Collecting model votes and generating final report...\n\n"
    
    try:
        # Finalize the evaluation
        final_report = jury_system.finalize(
            human_vote=selected_option,
            state=JURY_STATE
        )
        
        output_text += "\n" + "="*60 + "\n"
        output_text += "📊 FINAL EVALUATION REPORT\n"
        output_text += "="*60 + "\n\n"
        output_text += final_report
        
        return output_text
        
    except Exception as e:
        import traceback
        error_msg = f"❌ Error during finalization: {str(e)}\n\n"
        error_msg += traceback.format_exc()
        return error_msg

# Create Gradio interface
with gr.Blocks(title="Step 4: Voting") as step4_interface:
    gr.Markdown("# Step 4: Blind Voting")
    gr.Markdown("Review the anonymous evaluations and select the best one.")
    
    with gr.Row():
        with gr.Column():
            options_display = gr.Textbox(
                label="Anonymous Options",
                lines=15,
                max_lines=25,
                interactive=False
            )
            load_btn = gr.Button("Load Voting Options", variant="secondary")
        
        with gr.Column():
            vote_radio = gr.Radio(
                choices=[],
                label="Select the best evaluation"
            )
            submit_btn = gr.Button("Submit Vote", variant="primary")
            result_box = gr.Textbox(
                label="Final Report",
                lines=20,
                max_lines=30
            )
    
    # Load options when button clicked
    def load_options():
        text, radio = show_voting_options()
        return text, radio
    
    load_btn.click(
        fn=load_options,
        outputs=[options_display, vote_radio]
    )
    
    submit_btn.click(
        fn=submit_vote,
        inputs=vote_radio,
        outputs=result_box
    )

# Display the interface
step4_interface.launch(inline=True, share=False, debug=True)
